# Agribank OCR — GPU Worker (Google Colab)

## Chuẩn bị trên máy Windows (1 lần)
```powershell
cd colab
.\prepare_colab_zip.ps1
```
→ Tạo file `colab/ocr-service.zip`

## Trên Colab
1. **Runtime → Change runtime type → T4 GPU**
2. Run all cells theo thứ tự
3. Cell 2 sẽ hỏi upload `ocr-service.zip`
4. Copy **PUBLIC URL** + **TOKEN** vào FE (chế độ Google Colab)

In [ ]:
import os, secrets, shutil, subprocess

WORKER_TOKEN = secrets.token_urlsafe(16)
os.environ['PADDLE_USE_GPU'] = 'true'
os.environ['REMOTE_WORKER_TOKEN'] = WORKER_TOKEN
os.environ['OCR_WORKER_LABEL'] = 'Google Colab'
os.environ['PDF_DPI'] = '300'
os.environ['LOG_LEVEL'] = 'INFO'
os.environ['POPPLER_PATH'] = ''  # dùng poppler-utils hệ thống

# Cài Poppler sớm (cần cho PDF → ảnh)
subprocess.run(['apt-get', 'update', '-qq'], check=False)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'poppler-utils'], check=True)
assert shutil.which('pdfinfo'), 'poppler-utils chưa cài được'
print('Poppler OK:', shutil.which('pdfinfo'))

print('=' * 60)
print('TOKEN (copy vào FE Colab):', WORKER_TOKEN)
print('=' * 60)

In [ ]:
import os, zipfile, shutil
from pathlib import Path
from google.colab import files

SERVICE_DIR = Path('/content/ocr-service')

if not (SERVICE_DIR / 'app' / 'main.py').exists():
    print('>>> Chọn file ocr-service.zip (chạy prepare_colab_zip.ps1 trên máy Windows trước)')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('Chưa upload file zip')
    zip_name = next(iter(uploaded))
    extract_to = Path('/content/_zip_extract')
    if extract_to.exists():
        shutil.rmtree(extract_to)
    extract_to.mkdir()
    with zipfile.ZipFile(zip_name, 'r') as zf:
        for member in zf.namelist():
            # Sửa path Windows (app\main.py) → Linux (app/main.py)
            target = extract_to / member.replace('\\', '/')
            if member.endswith('/') or member.endswith('\\'):
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with zf.open(member) as src, open(target, 'wb') as dst:
                    dst.write(src.read())

    # Tìm thư mục gốc chứa app/main.py
    found = None
    if (extract_to / 'app' / 'main.py').exists():
        found = extract_to
    else:
        for p in extract_to.rglob('main.py'):
            if p.parent.name == 'app':
                found = p.parent.parent
                break
    if not found:
        print('Nội dung zip sau giải nén:')
        for p in sorted(extract_to.rglob('*'))[:30]:
            print(' ', p.relative_to(extract_to))
        raise RuntimeError('Zip không chứa app/main.py — chạy lại prepare_colab_zip.ps1 trên Windows')
    if SERVICE_DIR.exists():
        shutil.rmtree(SERVICE_DIR)
    shutil.move(str(found), str(SERVICE_DIR))
    print('Đã giải nén ocr-service →', SERVICE_DIR)
else:
    print('ocr-service đã có sẵn:', SERVICE_DIR)

%cd /content/ocr-service

# cloudflared tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# Cài Paddle GPU đúng CUDA cho Colab (tránh lỗi tensor GPU)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'paddlepaddle', 'paddlepaddle-gpu'], check=False)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'paddlepaddle-gpu==2.6.2',
    '-i', 'https://www.paddlepaddle.org.cn/packages/stable/cu118/',
], check=True)

# Cài dependencies còn lại (bỏ paddlepaddle-gpu trong requirements — đã cài ở trên)
req_lines = [
    line.strip() for line in open('requirements.txt')
    if line.strip() and not line.strip().startswith('#')
    and 'paddlepaddle' not in line.lower()
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + req_lines + ['setuptools'], check=True)

# Kiểm tra GPU Paddle
import paddle
paddle.utils.run_check()
print('Paddle device:', paddle.get_device())
print('Cài đặt xong.')

In [ ]:
import subprocess, threading, time, re, os, requests, shutil
from pathlib import Path

SERVICE_DIR = '/content/ocr-service'
assert Path(SERVICE_DIR, 'app', 'main.py').exists(), 'Thiếu ocr-service — chạy lại cell 2'

# Đảm bảo Poppler (kể cả khi bỏ qua cell 1 hoặc restart runtime)
if not shutil.which('pdfinfo'):
    subprocess.run(['apt-get', 'update', '-qq'], check=False)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'poppler-utils'], check=True)
assert shutil.which('pdfinfo'), 'poppler-utils chưa cài — chạy: apt-get install poppler-utils'
print('Poppler OK:', shutil.which('pdfinfo'))

env = os.environ.copy()
env['PADDLE_USE_GPU'] = 'true'
env['REMOTE_WORKER_TOKEN'] = WORKER_TOKEN
env['OCR_WORKER_LABEL'] = 'Google Colab'
env.pop('POPPLER_PATH', None)  # dùng poppler-utils trong PATH, không ép đường dẫn Windows

# Dừng process cũ nếu chạy lại cell
!pkill -f 'uvicorn app.main:app' 2>/dev/null || true
!pkill -f cloudflared 2>/dev/null || true
time.sleep(2)

def run_uvicorn():
    subprocess.Popen(
        ['uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8100'],
        cwd=SERVICE_DIR,
        env=env,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE,
    )

threading.Thread(target=run_uvicorn, daemon=True).start()

# Đợi uvicorn sẵn sàng (localhost)
headers = {'Authorization': f'Bearer {WORKER_TOKEN}'}
for i in range(40):
    try:
        r = requests.get('http://127.0.0.1:8100/health', headers=headers, timeout=3)
        if r.status_code == 200:
            print('Uvicorn OK:', r.json())
            break
    except Exception:
        pass
    time.sleep(2)
else:
    raise RuntimeError('Uvicorn không khởi động được — xem log ở cell trước')

# Mở tunnel public
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8100', '--no-autoupdate'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

public_url = None
url_pattern = re.compile(r'https://[a-z0-9-]+\.trycloudflare\.com')
deadline = time.time() + 60
while time.time() < deadline and public_url is None:
    line = tunnel.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    m = url_pattern.search(line)
    if m:
        public_url = m.group(0)
        break

print()
print('=' * 60)
print('PUBLIC URL (dán vào FE):', public_url or '(chưa lấy được — xem output cloudflared)')
print('TOKEN:', WORKER_TOKEN)
print('=' * 60)
print('Giữ notebook này chạy. Đóng = worker ngắt.')
print('Kiểm tra từ máy local: nút "Kiểm tra Colab" trên FE')